# Loop-Lingua T4 pilot (not run)

96 generations: 24 matched prompts in English, Hindi, Telugu, and romanized Telugu.
Model: `Qwen/Qwen3.5-4B` on one T4, greedy decoding.

This notebook was executed only as a CPU dry-run. It does not contain a tokens/second measurement. The cell that would call `model.generate` is not run.


In [1]:
from loop_lingua.pilot import PilotConfig, describe_plan, plan_go_nogo

config = PilotConfig()
print(describe_plan(config))
print()
pending = plan_go_nogo(None, config, basis="none")
print("decision:", pending.decision)
print(pending.reasons[0])


Pilot: 96 generations = 24 prompts × 4 languages.
Languages: en, hi, te, te_roman.
Pilot cap: 512 new tokens. Full-study cap used for the budget: 2048.
Full study: 1200 sequences, upper bound 2457600 new tokens.
Budget: 8 h on one T4. Health floor: 15 tok/s.
Model id (not loaded here): Qwen/Qwen3.5-4B.
This process has not generated any tokens.

decision: pending
No tokens/second figure yet. Run the 96-generation pilot on a T4.


## What-if arithmetic

The next cell plugs in a made-up rate so the formula is visible. It is not a result from a T4.


In [2]:
assumed = plan_go_nogo(40.0, config, basis="assumed")
print("decision:", assumed.decision, "basis:", assumed.basis)
print(f"upper-bound hours: {assumed.estimated_full_hours:.2f}")
print("new tokens that fit in the budget:", assumed.max_new_tokens_that_fit)
for caveat in assumed.caveats:
    print("-", caveat)


decision: replan basis: assumed
upper-bound hours: 17.07
new tokens that fit in the budget: 960
- Hours assume tokens per second stay constant out to full_max_new_tokens. Long contexts on a T4 are slower than a short pilot, so a 'go' is an upper bound on speed.
- The token total assumes every sequence emits full_max_new_tokens. Early stops make the run shorter.
- Decoding in this plan is greedy. Liquid's Antidoom pipeline used temperature 0.01.


## On the T4

Do not run the next cell on a CPU worker. It is the shape of the call. `transformers_greedy_generate` was not executed while this package was built. Confirm that the Qwen3.5 class exposes `generate` and `device` before trusting it. Qwen3.5-4B is a multimodal checkpoint; `AutoModelForCausalLM` may be the wrong class.

```python
# NOT RUN
from loop_lingua.prompts import read_jsonl
from loop_lingua.layout import CORPUS_DIR
from loop_lingua.pilot import PilotJob, run_harness, transformers_greedy_generate, plan_go_nogo

rows = read_jsonl(CORPUS_DIR / "pilot_96.jsonl")
jobs = [PilotJob(r["item_id"], r["language"], r["prompt"]) for r in rows]
# model, tokenizer = load on the T4, bf16, device_map="auto"
# report = run_harness(jobs, lambda job: transformers_greedy_generate(model, tokenizer, job, max_new_tokens=512))
# decision = plan_go_nogo(report.tokens_per_second, basis="measured")
```

Score the resulting texts with `detect` (D0–D4) and `measure_drift`. Pass characters-per-token from `measure_fertility` on the prompts, not a default, into D4.
